In [1]:
import numpy as np
from plotly.io import show

from skfolio import MultiPeriodPortfolio, Population, Portfolio
from skfolio.datasets import load_sp500_dataset
from skfolio.model_selection import WalkForward, cross_val_predict
from skfolio.optimization import MeanRisk, ObjectiveFunction
from skfolio.preprocessing import prices_to_returns

prices = load_sp500_dataset()
prices = prices[["AAPL", "GE", "JPM"]]

X = prices_to_returns(prices)

In [2]:
model = MeanRisk(objective_function=ObjectiveFunction.MAXIMIZE_UTILITY)
model.fit(X)
model.weights_

array([6.17733231e-01, 3.78774141e-09, 3.82266765e-01])

In [3]:
transaction_costs = {"AAPL": 0.01 / 21, "GE": 0.005 / 21, "JPM": 0.002 / 21}
# Same as transaction_costs = np.array([0.01, 0.005, 0.002]) / 21

In [4]:
model_tc = MeanRisk(
    objective_function=ObjectiveFunction.MAXIMIZE_UTILITY,
    transaction_costs=transaction_costs,
)
model_tc.fit(X)
model_tc.weights_

array([4.11868007e-01, 1.40979808e-07, 5.88131852e-01])

In [5]:
model_tc.weights_ - model.weights_

array([-2.05865225e-01,  1.37192067e-07,  2.05865087e-01])

In [6]:
model_tc2 = MeanRisk(
    objective_function=ObjectiveFunction.MAXIMIZE_UTILITY,
    transaction_costs=transaction_costs,
    previous_weights=np.ones(3) / 3,
)
model_tc2.fit(X)
model_tc2.weights_

array([0.33333336, 0.3333332 , 0.33333345])

In [7]:
model_tc2.weights_ - model.weights_

array([-0.28439988,  0.3333332 , -0.04893332])

In [8]:
#multi period portfolio
holding_period = 60
fitting_period = 60
cv = WalkForward(train_size=fitting_period, test_size=holding_period)

In [9]:
transaction_costs = np.array([0.01, 0.005, 0.002]) / holding_period

In [10]:
model = MeanRisk(objective_function=ObjectiveFunction.MAXIMIZE_UTILITY)
# pred1 is a MultiPeriodPortfolio
pred1 = cross_val_predict(model, X, cv=cv, n_jobs=-1)
pred1.name = "pred1"

In [11]:
pred2 = MultiPeriodPortfolio(name="pred2")
previous_weights = None
for portfolio in pred1:
    new_portfolio = Portfolio(
        X=portfolio.X,
        weights=portfolio.weights,
        previous_weights=previous_weights,
        transaction_costs=transaction_costs,
    )
    previous_weights = portfolio.weights
    pred2.append(new_portfolio)

In [12]:
pred3 = MultiPeriodPortfolio(name="pred3")

model.set_params(transaction_costs=transaction_costs)
previous_weights = None
for train, test in cv.split(X):
    X_train = X.take(train)
    X_test = X.take(test)
    model.set_params(previous_weights=previous_weights)
    model.fit(X_train)
    portfolio = model.predict(X_test)
    pred3.append(portfolio)
    previous_weights = model.weights_

In [13]:
population = Population([pred1, pred2, pred3])
fig = population.plot_cumulative_returns()
show(fig)

If we exclude the unrealistic prediction without TC, we notice that the model fitted with TC outperforms the model fitted without TC.